In [1]:
!pip install scikit-learn

In [2]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd
df=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# 데이터 준비
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']

data = df[features].copy()

data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
data['Age'] = data['Age'].fillna(data['Age'].median())
data['Fare'] = data['Fare'].fillna(data['Fare'].median())

X = data
y = df['Survived']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 파이프라인
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

# 튜닝할 파라미터
param_grid = {
    'svc__kernel': ['rbf', 'linear', 'poly'],
    'svc__C': [0.1, 1, 3, 10, 30, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svc__degree': [2, 3, 4]  # poly일 때만 의미 있음
}

# Grid Search
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("best params:", grid.best_params_)
print("best cv score:", grid.best_score_)

best_model = grid.best_estimator_
pred = best_model.predict(X_valid)

print("valid accuracy:", accuracy_score(y_valid, pred))
print(classification_report(y_valid, pred))

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
best params: {'svc__C': 30, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'rbf'}
best cv score: 0.8202895695853443
valid accuracy: 0.7932960893854749
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179



In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV

# -----------------------
# train data
# -----------------------
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']

X_train_df = df[features].copy()
y = df['Survived']

X_train_df['Sex'] = X_train_df['Sex'].map({'male': 0, 'female': 1})
X_train_df['Age'] = X_train_df['Age'].fillna(df['Age'].median())
X_train_df['Fare'] = X_train_df['Fare'].fillna(df['Fare'].median())

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

param_grid = [
    {
        'svc__kernel': ['linear'],
        'svc__C': [0.1, 1, 3, 10, 30]
    },
    {
        'svc__kernel': ['rbf'],
        'svc__C': [0.1, 1, 3, 10, 30],
        'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1]
    }
]

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_df, y)
best_model = grid.best_estimator_

print("best params:", grid.best_params_)
print("best cv score:", grid.best_score_)

# -----------------------
# test data
# -----------------------
dt = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

T = dt[features].copy()
T['Sex'] = T['Sex'].map({'male': 0, 'female': 1})
T['Age'] = T['Age'].fillna(df['Age'].median())
T['Fare'] = T['Fare'].fillna(df['Fare'].median())

pred = best_model.predict(T)

submission = pd.DataFrame({
    'PassengerId': dt['PassengerId'],
    'Survived': pred
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print("saved: /kaggle/working/submission.csv")

best params: {'svc__C': 3, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
best cv score: 0.830525390747599
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         0
saved: /kaggle/working/submission.csv
